# Closure-Certified Temporal Deduction on Real Text — Demo

**Artifact:** *Powered closure-certified temporal deduction on real text: H1+H2 CONFIRM at n=600*

This notebook is a small, fully-offline demo of the **Closure-Certified Text-to-Logic Deduction Module**.
The research question: can **iterated path-consistency closure** over *span-local* LLM reads of the
constituent edges of a temporal graph **recover deduction-required relations the local reads never saw
directly** — beating purely-neural baselines and cutting confident hallucinations?

**Our method — Mode-A.** For a held-out query edge `A?C`, the LLM reads only the *local* pairwise temporal
relations along constituent paths (e.g. `A?B`, `B?C`). We then run **PC-2 iterated path-consistency closure**
(`engine.pc2_full`) in the **convex point start-point algebra** (PC-complete, exact). Mode-A *answers* iff the
closure narrows the query edge to a single coarse relation, else it **ABSTAINs**. If the local reads are
mutually inconsistent, closure emits a **Mode-B inconsistency certificate** (and abstains).

**Baselines** in the same pipeline / coverage object: `naive` single-pass intersection, `raw` local LLM
(forced single relation from the query edge's own read), `Path-of-Thoughts` (per-path composition + modal vote,
no cross-path intersection), and `self-consistency` (k=5 paraphrase votes).

**What this demo does (no API keys, no network to LLMs, no SWI-Prolog needed):**
1. Loads the **real closure engine** (`engine.py`, reused verbatim) — the THEOREM-grade core.
2. Reproduces the two **SWI-Prolog-discharged worked examples** directly with the Python engine:
   a Mode-A narrowing (→ `ANSWER(before)`) and a Mode-B inconsistency collapse (→ `INCONSISTENT`).
3. Recomputes the **real-text head-to-head metrics** (coverage, selective accuracy, H2 confident-wrong
   reduction) from a curated subset of the powered run's per-query predictions.

> The heavy lifting in the full experiment is the LLM elicitation of local reads (≈3,900 reads, 600 queries
> across NarrativeTime + TDDMan). Those reads are **pre-computed and cached** in the released data, so this
> demo reproduces the closure mechanism + the metrics in seconds. Headline numbers for the full `n=600` run
> are printed alongside for context.

In [ ]:
# --- Dependencies (Colab-safe install pattern) ---
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# This demo only needs numpy + matplotlib, both pre-installed on Colab.
# On Colab: skip (use Colab's versions). Locally: install at Colab's exact versions.
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'matplotlib==3.10.0')

In [ ]:
# --- Imports ---
# stdlib used by the reused closure engine + the demo
import itertools
import json
import os
import urllib.request
from collections import deque, defaultdict
from typing import Iterable

import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# --- Data loading (GitHub URL with local fallback, Colab-compatible) ---
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-40a89b-no-derivation-no-relation-a-closure-cert/main/round-3/experiment-2/demo/mini_demo_data.json"

def load_data():
    try:
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()

examples = data["datasets"][0]["examples"]
worked   = data["worked_examples"]
headline = data["headline_findings"]

print("corpus        :", data["datasets"][0]["dataset"])
print("queries loaded:", len(examples))
print("worked progs  :", [w["kind"] for w in worked])
print("full-run verdict:", headline["verdict"],
      "| H1(vs PoT & SC):", headline["H1_confirm_vs_PoT_and_SC"],
      "| H2(halluc red):", headline["H2_confirm_hallucination_reduction"])

## Configuration

All tunable knobs live here. They start at small demo values; comments give the **full-run** values from
the powered `n=600` experiment. Everything in this demo is closure + arithmetic over pre-computed reads, so
it runs in well under a second even at the full curated size.

In [ ]:
# ---- Demo configuration (small values; full-run values in comments) ----
CORPUS        = "narrativetime"   # one corpus for the demo (full run: narrativetime + tddman)
N_EXAMPLES    = 100               # curated queries to score (full run: 600 across 2 corpora)
N_BOOT        = 2000              # doc-clustered bootstrap reps for the H2 CI (full run BOOT_B = 2000)
SEED          = 20260617          # pre-registered seed
ALPHA         = 0.05              # 95% CIs
RECALL_GATE_POINT = 0.90          # GOLD-ONLY read-soundness gate for the PC-complete point arm
H2_MIN_EFFECT = 0.05              # confident-wrong must drop >= 5 pts ABS vs raw (pre-registered)

# The five coarse temporal labels used everywhere (POINT arm).
COARSE_LABELS = ["before", "after", "simultaneous", "includes", "is_included"]

examples = examples[:N_EXAMPLES]
print(f"scoring {len(examples)} {CORPUS} deduction queries")

## Part 1 — The closure engine (`engine.py`, reused verbatim)

The qualitative-constraint-network (QCN) closure engine, copied unchanged from the artifact. Two algebras are
built **programmatically** (no external table files):

* **POINT** — the convex point algebra over event *start-points* `{<,=,>}`. It is **PC-complete**, so its
  inconsistency certificate is **exact**. The five coarse temporal labels map bijectively to the five convex
  point relations: `before={<}`, `after={>}`, `simultaneous={=}`, `includes={<,=}`, `is_included={=,>}`; the
  universe `{<,=,>}` means *no commitment*.
* **ALLEN** — the 13-relation interval algebra, composition generated by endpoint enumeration (sound but
  incomplete; included for completeness, not used by the POINT-arm demo).

Closure operators: `close_triangle` (length-2 narrowing), **`pc2_full`** (Mackworth PC-2 worklist closure to
fixpoint — *our method*), and `naive_single_pass` (one pass, no re-propagation — a baseline).

In [ ]:
#!/usr/bin/env python3
"""Self-contained qualitative-constraint-network (QCN) closure engine.

Two algebras built PROGRAMMATICALLY (no external table files), each cross-checked
against the verified GQR composition cells from the implementation dossier:

  * POINT  -- convex point algebra over event START-points {<,=,>} (PC COMPLETE;
              EXACT inconsistency certificate). The ONLY non-convex relation is
              `{<,>}` (`!=`); per the dossier widening rule it is WIDENED to the
              universal `{<,=,>}` to keep path-consistency complete, and every
              widening is COUNTED (bite lost by convex restriction).
  * ALLEN  -- Allen interval algebra (13 base relations). Composition is generated
              by the ENDPOINT method (enumerate weak orders of the six interval
              endpoints) and unit-tested against the dossier's GQR cells. Full Allen
              PC is sound but INCOMPLETE -> closure-detectable inconsistency is a
              LOWER BOUND there.

Closure operators:
  * close_triangle      -- the per-triangle length-2 path narrowing used for the
                           frontier metrics (path = compose(AB,BC); inter = path & AC).
  * pc2_full            -- Mackworth PC-2 worklist closure to fixpoint (OUR METHOD).
  * naive_single_pass   -- BASELINE: one pass of length-2 path intersections at the
                           query edge, no fixpoint / no re-propagation
                           ("Path-of-Thoughts + one intersection step").
"""


# ----------------------------------------------------------------------------------
# Allen-13 base relations (qualreas-compatible symbols) and their endpoint geometry.
# Interval X = (Xs, Xe) with Xs < Xe. relation(X, Y) defined on (Xs, Xe, Ys, Ye).
# ----------------------------------------------------------------------------------
ALLEN_BASE = ["B", "BI", "D", "DI", "O", "OI", "M", "MI", "S", "SI", "F", "FI", "E"]
ALLEN_CONVERSE = {"B": "BI", "BI": "B", "D": "DI", "DI": "D", "O": "OI", "OI": "O",
                  "M": "MI", "MI": "M", "S": "SI", "SI": "S", "F": "FI", "FI": "F", "E": "E"}


def _allen_rel(xs: int, xe: int, ys: int, ye: int) -> str | None:
    """Atomic Allen relation of interval X=(xs,xe) to Y=(ys,ye) from endpoint ranks.

    Assumes proper intervals (xs<xe, ys<ye). Returns one of the 13 base symbols.
    """
    if not (xs < xe and ys < ye):
        return None
    if xs == ys and xe == ye:
        return "E"
    if xe < ys:
        return "B"
    if ye < xs:
        return "BI"
    if xe == ys:
        return "M"
    if ye == xs:
        return "MI"
    if xs == ys:
        return "S" if xe < ye else "SI"
    if xe == ye:
        return "F" if xs > ys else "FI"
    if xs < ys and ye < xe:
        return "DI"
    if ys < xs and xe < ye:
        return "D"
    if xs < ys < xe < ye:
        return "O"
    if ys < xs < ye < xe:
        return "OI"
    return None  # unreachable for proper intervals


def _build_allen_compose() -> dict[tuple[str, str], frozenset]:
    """Generate the Allen base x base composition table via endpoint enumeration.

    Enumerate every rank assignment of the 6 endpoints (As,Ae,Bs,Be,Cs,Ce) in 0..5
    (ties allowed; only relative order matters). For each that yields proper intervals,
    record comp[r(A,B)][r(B,C)] |= {r(A,C)}.
    """
    comp: dict[tuple[str, str], set] = {(a, b): set() for a in ALLEN_BASE for b in ALLEN_BASE}
    for asg in itertools.product(range(6), repeat=6):
        As, Ae, Bs, Be, Cs, Ce = asg
        if not (As < Ae and Bs < Be and Cs < Ce):
            continue
        rab = _allen_rel(As, Ae, Bs, Be)
        rbc = _allen_rel(Bs, Be, Cs, Ce)
        rac = _allen_rel(As, Ae, Cs, Ce)
        if rab is None or rbc is None or rac is None:
            continue
        comp[(rab, rbc)].add(rac)
    return {k: frozenset(v) for k, v in comp.items()}


# ----------------------------------------------------------------------------------
# Point algebra over start-points.
# ----------------------------------------------------------------------------------
POINT_BASE = ["<", "=", ">"]
POINT_CONVERSE = {"<": ">", "=": "=", ">": "<"}
# GQR point.comp (verified): = is identity; <o<={<}; >o>={>}; <o>=>o<=universal.
POINT_COMPOSE = {
    ("=", "="): frozenset({"="}),
    ("<", "="): frozenset({"<"}), ("=", "<"): frozenset({"<"}),
    (">", "="): frozenset({">"}), ("=", ">"): frozenset({">"}),
    ("<", "<"): frozenset({"<"}),
    (">", ">"): frozenset({">"}),
    ("<", ">"): frozenset({"<", "=", ">"}),
    (">", "<"): frozenset({"<", "=", ">"}),
}
POINT_NONCONVEX = frozenset({"<", ">"})  # the only non-convex point relation (`!=`)


class Algebra:
    """A qualitative calculus with relation sets stored as frozensets of base symbols."""

    def __init__(self, name: str, base: list[str], converse: dict[str, str],
                 compose_bb: dict[tuple[str, str], frozenset], identity: frozenset,
                 convex_widen: frozenset | None = None):
        self.name = name
        self.base = list(base)
        self.universe = frozenset(base)
        self.empty = frozenset()
        self.identity = frozenset(identity)
        self._conv = dict(converse)
        self._comp = dict(compose_bb)
        # `convex_widen` (point algebra only): the unique non-convex relation that must
        # be widened to the universal set to keep PC complete. None => no widening.
        self._nonconvex = convex_widen

    # ---- set ops -----------------------------------------------------------------
    def converse(self, s: frozenset) -> frozenset:
        return frozenset(self._conv[r] for r in s)

    def compose(self, a: frozenset, b: frozenset) -> frozenset:
        if not a or not b:
            return self.empty
        out: set = set()
        for x in a:
            for y in b:
                out |= self._comp[(x, y)]
        return frozenset(out)

    def is_nonconvex(self, s: frozenset) -> bool:
        return self._nonconvex is not None and s == self._nonconvex

    def widen(self, s: frozenset) -> tuple[frozenset, bool]:
        """Return (possibly-widened set, fired?). Widening only happens for the
        unique non-convex relation of a convex algebra (point: `{<,>}` -> universe)."""
        if self._nonconvex is not None and s == self._nonconvex:
            return self.universe, True
        return s, False

    def label(self, s: frozenset) -> str:
        if not s:
            return "EMPTY"
        if s == self.universe:
            return "UNIVERSE"
        return "|".join(r for r in self.base if r in s)


def build_point_algebra() -> Algebra:
    return Algebra("POINT", POINT_BASE, POINT_CONVERSE, POINT_COMPOSE,
                   frozenset({"="}), convex_widen=POINT_NONCONVEX)


def build_allen_algebra() -> Algebra:
    return Algebra("ALLEN", ALLEN_BASE, ALLEN_CONVERSE, _build_allen_compose(),
                   frozenset({"E"}), convex_widen=None)


# ----------------------------------------------------------------------------------
# Triangle closure (frontier metric primitive)
# ----------------------------------------------------------------------------------
def close_triangle(alg: Algebra, ab: frozenset, bc: frozenset, ac: frozenset):
    """Length-2 path A-B-C narrowing the query edge A-C.

    path  = compose(ab, bc)                    (with convex widening applied + counted)
    inter = path & ac                          (widening applied + counted)
    Returns dict: path, inter, empty(collapse), singleton, n_widen.
    """
    n_widen = 0
    path = alg.compose(ab, bc)
    path, w = alg.widen(path)
    n_widen += int(w)
    inter = path & ac
    inter, w = alg.widen(inter)
    n_widen += int(w)
    return {
        "path": path,
        "inter": inter,
        "empty": len(inter) == 0,
        "singleton": len(inter) == 1,
        "n_widen": n_widen,
    }


# ----------------------------------------------------------------------------------
# Full QCN + closure variants (reused by whole-document consistency checks)
# ----------------------------------------------------------------------------------
class QCN:
    """Constraint network: dense node list, edges = relation-set frozensets.

    Missing edge => universe. Invariants on set_edge: M[j][i]==converse(M[i][j]),
    M[i][i]==identity.
    """

    def __init__(self, alg: Algebra, nodes: list):
        self.alg = alg
        self.nodes = list(nodes)
        self.n = len(self.nodes)
        self.index = {nd: i for i, nd in enumerate(self.nodes)}
        U = alg.universe
        self.M = [[U] * self.n for _ in range(self.n)]
        for i in range(self.n):
            self.M[i][i] = alg.identity
        self.nbrs: list[set] = [set() for _ in range(self.n)]

    def set_edge(self, i: int, j: int, s: frozenset) -> None:
        if i == j:
            return
        self.M[i][j] = s
        self.M[j][i] = self.alg.converse(s)
        if s != self.alg.universe:
            self.nbrs[i].add(j); self.nbrs[j].add(i)
        else:
            self.nbrs[i].discard(j); self.nbrs[j].discard(i)

    def get(self, i: int, j: int) -> frozenset:
        return self.M[i][j]

    def known_edges(self) -> list[tuple[int, int]]:
        U = self.alg.universe
        return [(i, j) for i in range(self.n) for j in range(i + 1, self.n) if self.M[i][j] != U]


def pc2_full(qcn: QCN):
    """OUR METHOD: Mackworth PC-2 worklist closure to fixpoint.

    Returns (consistent: bool, n_fired). Empty edge => inconsistent (Mode-B certificate).
    Convex widening (point algebra) applied to every refined edge and absorbed silently
    (it can only enlarge a set, never empties it).
    """
    alg = qcn.alg
    U = alg.universe
    M = qcn.M
    nbrs = qcn.nbrs
    Q = deque()
    inq = set()
    for (i, j) in qcn.known_edges():
        Q.append((i, j)); inq.add((i, j))
        Q.append((j, i)); inq.add((j, i))
    n_fired = 0

    def enqueue(a, b):
        if (a, b) not in inq:
            inq.add((a, b)); Q.append((a, b))

    while Q:
        i, j = Q.popleft(); inq.discard((i, j))
        rij = M[i][j]
        if rij == U:
            continue
        for k in list(nbrs[j]):
            if k == i:
                continue
            comp = alg.compose(rij, M[j][k])
            new = M[i][k] & comp
            new, _ = alg.widen(new)
            if new != M[i][k]:
                if not new:
                    return False, n_fired
                M[i][k] = new; M[k][i] = alg.converse(new)
                nbrs[i].add(k); nbrs[k].add(i)
                n_fired += 1
                enqueue(i, k); enqueue(k, i)
        for k in list(nbrs[i]):
            if k == j:
                continue
            comp = alg.compose(M[k][i], rij)
            new = M[k][j] & comp
            new, _ = alg.widen(new)
            if new != M[k][j]:
                if not new:
                    return False, n_fired
                M[k][j] = new; M[j][k] = alg.converse(new)
                nbrs[k].add(j); nbrs[j].add(k)
                n_fired += 1
                enqueue(k, j); enqueue(j, k)
    return True, n_fired


def naive_single_pass(qcn: QCN, u: int, v: int) -> frozenset:
    """BASELINE: single pass of length-2 path compositions at the query edge (u,v).

    Intersects compose(R(u,w), R(w,v)) over intermediate w with informative u-w & w-v.
    NO fixpoint, NO re-propagation.
    """
    alg = qcn.alg
    U = alg.universe
    M = qcn.M
    R = U
    for w in qcn.nbrs[u]:
        if w in (u, v):
            continue
        if M[w][v] != U:
            R = R & alg.compose(M[u][w], M[w][v])
            R, _ = alg.widen(R)
            if not R:
                return alg.empty
    return R

### Coarse ↔ point maps

The bijection between the five coarse temporal labels and the five convex point relations, plus the small
helpers used to read a closed query edge back into a coarse label. These are copied from `corpora.py` /
`method.py`.

In [ ]:
# coarse label -> POINT start-point set  (corpora.py)
COARSE_TO_POINT = {
    "before": frozenset({"<"}), "after": frozenset({">"}),
    "simultaneous": frozenset({"="}),
    "includes": frozenset({"<", "="}), "is_included": frozenset({"=", ">"}),
}
# the 5 coarse labels <-> the 5 convex point relations (exact bijection)  (method.py)
POINT_SET_TO_COARSE = {
    frozenset({"<"}): "before", frozenset({">"}): "after",
    frozenset({"="}): "simultaneous", frozenset({"<", "="}): "includes",
    frozenset({"=", ">"}): "is_included",
}
COARSE_CONVERSE = {"before": "after", "after": "before", "simultaneous": "simultaneous",
                   "includes": "is_included", "is_included": "includes"}

PT = build_point_algebra()   # the PC-complete point algebra instance used by the method

def point_to_coarse(R):
    return POINT_SET_TO_COARSE.get(frozenset(R))

# atomic point relation symbols used in the worked-example Prolog programs
PROLOG_ATOM_TO_POINT = {"lt": frozenset({"<"}), "eq": frozenset({"="}), "gt": frozenset({">"})}

print("point algebra base:", PT.base)
print("< o > =", sorted(PT.compose(frozenset({'<'}), frozenset({'>'}))), "(universe -> no commitment)")
print("< o < =", sorted(PT.compose(frozenset({'<'}), frozenset({'<'}))), "(transitive -> before)")

## Part 2 — Reproducing the SWI-Prolog worked examples with the Python engine

The full experiment emits runnable SWI-Prolog trace programs and discharges them with `swipl` for human
auditing. Here we **parse the `rel/3` local-read facts straight out of those released programs**, build the
QCN, run the **same** closure (`pc2_full`) the engine uses, and check the Python verdict matches the
discharged `swipl` verdict — no Prolog install required.

* **Mode-A narrowing:** the query edge is *held out* (left at universe); closure over the local reads narrows
  it to a single relation → `ANSWER`. Expected: `before`.
* **Mode-B collapse:** the local reads are mutually inconsistent; closure empties an edge → an inconsistency
  certificate → Mode-B `ABSTAIN`.

In [ ]:
import re

def parse_rel_facts(program):
    """Pull rel(A,B,atom). local-read facts from a worked-example Prolog program."""
    facts = []
    for a, b, atom in re.findall(r"rel\(([^,]+),([^,]+),(lt|eq|gt)\)\.", program):
        facts.append((a.strip(), b.strip(), atom.strip()))
    return facts

def run_engine_on_worked(w):
    """Rebuild the QCN from the released local reads and run pc2_full (query edge held out)."""
    facts = parse_rel_facts(w["program"])
    qx, qy = w["query"] if w.get("query") else (None, None)
    nodes = sorted({n for (a, b, _) in facts for n in (a, b)})
    qcn = QCN(PT, nodes)
    for (a, b, atom) in facts:
        if qx is not None and tuple(sorted((a, b))) == tuple(sorted((qx, qy))):
            continue  # never set the query edge -> it must be DEDUCED by closure
        qcn.set_edge(qcn.index[a], qcn.index[b], PROLOG_ATOM_TO_POINT[atom])
    consistent, n_fired = pc2_full(qcn)
    if not consistent:
        return {"verdict": "INCONSISTENT", "coarse": None, "n_fired": n_fired}
    if qx is None:
        return {"verdict": "CONSISTENT", "coarse": None, "n_fired": n_fired}
    R = qcn.get(qcn.index[qx], qcn.index[qy])
    coarse = point_to_coarse(R)
    return {"verdict": "ANSWER" if coarse else "ABSTAIN",
            "coarse": coarse, "point_set": sorted(R), "n_fired": n_fired}

for w in worked:
    print("=" * 78)
    print("WORKED EXAMPLE:", w["kind"])
    print("  swipl stdout       :", repr(w["swipl_stdout"]))
    print("  swipl verdict      :", w["swipl_verdict"], "| gold coarse:", w["gold_coarse"])
    res = run_engine_on_worked(w)
    print("  python engine      :", res)
    match = (res["verdict"] == w["swipl_verdict"])
    print("  python == swipl ?  :", match,
          "(released swipl_matches_python_checker =", w.get("swipl_matches_python_checker"), ")")
    assert match, "engine verdict disagrees with discharged swipl verdict!"
print("=" * 78)
print("Both worked programs reproduced by the Python engine. ✓")

## Part 3 — Real-text head-to-head metrics

We reconstruct each method's per-query record from the released predictions (`predict_modeA`, `predict_naive`,
`predict_raw`, `predict_pot`, `predict_sc`) against the gold relation `output`:

* `answered`  = prediction is not `ABSTAIN`
* `correct`   = prediction equals gold (only defined when answered)
* **coverage**          = fraction of queries answered
* **selective accuracy** = accuracy *among answered* queries (the fair quality metric at a given coverage)
* **confident-wrong (H2)** = fraction of answered predictions that are wrong = `1 − selective accuracy`

**H2** compares Mode-A's confident-wrong rate against `raw` with a **doc-clustered paired bootstrap** on the
reduction (resampling documents, the unit of dependence), exactly as in `method.py`.

In [ ]:
METHODS = ["modeA", "naive", "raw", "pot", "sc"]
PRETTY  = {"modeA": "Mode-A (ours)", "naive": "naive", "raw": "raw LLM",
           "pot": "Path-of-Thoughts", "sc": "self-consistency"}

def coverage_selacc(exs, method):
    """coverage, selective accuracy, n_answered for one method over the query pool."""
    ans = [e for e in exs if e["predict_" + method] != "ABSTAIN"]
    n_ans = len(ans)
    n_cor = sum(1 for e in ans if e["predict_" + method] == e["output"])
    cov = n_ans / len(exs) if exs else 0.0
    sel = (n_cor / n_ans) if n_ans else float("nan")
    return cov, sel, n_ans

summary = {}
for m in METHODS:
    cov, sel, n_ans = coverage_selacc(examples, m)
    summary[m] = {"coverage": cov, "selective_acc": sel,
                  "confident_wrong": (1 - sel) if n_ans else float("nan"), "n_answered": n_ans}

# ---- H2: Mode-A vs raw confident-wrong, doc-clustered paired bootstrap on the reduction ----
def h2_reduction_bootstrap(exs, B, seed, alpha):
    by_doc = defaultdict(lambda: {"modeA": [], "raw": []})
    for e in exs:
        d = e["metadata_docid"]
        for m in ("modeA", "raw"):
            if e["predict_" + m] != "ABSTAIN":
                by_doc[d][m].append(int(e["predict_" + m] != e["output"]))  # 1 == confident-wrong
    docs = list(by_doc)
    rng = np.random.default_rng(seed)
    reds = []
    for _ in range(B):
        pick = [docs[i] for i in rng.integers(0, len(docs), len(docs))]
        ma = [x for d in pick for x in by_doc[d]["modeA"]]
        ra = [x for d in pick for x in by_doc[d]["raw"]]
        if ma and ra:
            reds.append(np.mean(ra) - np.mean(ma))
    lo, hi = np.quantile(reds, [alpha / 2, 1 - alpha / 2])
    return float(np.quantile(reds, 0.5)), (float(lo), float(hi)), float(np.mean([r <= 0 for r in reds]))

cw_modeA = summary["modeA"]["confident_wrong"]
cw_raw   = summary["raw"]["confident_wrong"]
point_reduction = cw_raw - cw_modeA
med_red, (red_lo, red_hi), boot_p = h2_reduction_bootstrap(examples, N_BOOT, SEED, ALPHA)

print(f"{'method':18s} {'coverage':>9s} {'sel.acc':>9s} {'conf-wrong':>11s} {'n_ans':>6s}")
for m in METHODS:
    s = summary[m]
    print(f"{PRETTY[m]:18s} {s['coverage']:>9.3f} {s['selective_acc']:>9.3f} "
          f"{s['confident_wrong']:>11.3f} {s['n_answered']:>6d}")
print()
print(f"H2 confident-wrong: Mode-A {cw_modeA:.3f}  vs  raw {cw_raw:.3f}")
print(f"H2 reduction = {point_reduction:.3f}  (doc-clustered boot 95% CI [{red_lo:.3f}, {red_hi:.3f}], "
      f"p(reduction<=0)={boot_p:.4f})")
print(f"H2 passes pre-registered >= {H2_MIN_EFFECT} ABS drop ? {point_reduction >= H2_MIN_EFFECT}")

## Part 4 — Visualization

* **Left:** coverage vs selective accuracy per method. Mode-A answers *few* queries (low coverage) but the
  ones it does answer are *more accurate* (high selective accuracy) — it abstains instead of hallucinating.
* **Right:** the H2 confident-wrong (hallucination) rate, Mode-A vs raw, with the closure-driven reduction.

The demo numbers are computed on a 100-query single-corpus subset, so they differ from the powered `n=600`
headline (printed for context) but reproduce the same qualitative story.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# ---- Left: coverage vs selective accuracy ----
x = np.arange(len(METHODS))
covs = [summary[m]["coverage"] for m in METHODS]
sels = [summary[m]["selective_acc"] for m in METHODS]
w = 0.38
ax1.bar(x - w / 2, covs, w, label="coverage", color="#9ecae1")
ax1.bar(x + w / 2, sels, w, label="selective accuracy", color="#3182bd")
ax1.set_xticks(x); ax1.set_xticklabels([PRETTY[m] for m in METHODS], rotation=20, ha="right")
ax1.set_ylim(0, 1.05); ax1.set_ylabel("rate")
ax1.set_title("Coverage vs selective accuracy (per method)")
ax1.legend(loc="upper center")
for xi, (c, s) in enumerate(zip(covs, sels)):
    ax1.text(xi - w / 2, c + 0.01, f"{c:.2f}", ha="center", va="bottom", fontsize=8)
    if s == s:
        ax1.text(xi + w / 2, s + 0.01, f"{s:.2f}", ha="center", va="bottom", fontsize=8)

# ---- Right: H2 confident-wrong (Mode-A vs raw) ----
bars = ax2.bar(["Mode-A (ours)", "raw LLM"], [cw_modeA, cw_raw],
               color=["#31a354", "#de2d26"])
ax2.set_ylim(0, max(cw_modeA, cw_raw) * 1.35 + 0.05)
ax2.set_ylabel("confident-wrong (hallucination) rate")
ax2.set_title("H2: confident-wrong rate — Mode-A abstains instead of hallucinating")
for b, v in zip(bars, [cw_modeA, cw_raw]):
    ax2.text(b.get_x() + b.get_width() / 2, v + 0.01, f"{v:.3f}", ha="center", va="bottom")
ax2.annotate(f"reduction = {point_reduction:.3f}\n95% CI [{red_lo:.2f}, {red_hi:.2f}]",
             xy=(0, cw_modeA), xytext=(0.5, max(cw_modeA, cw_raw) * 1.18),
             ha="center", fontsize=10,
             bbox=dict(boxstyle="round", fc="#ffffcc", ec="gray"))
plt.tight_layout()
plt.savefig("demo_results.png", dpi=110, bbox_inches="tight")
plt.show()

# ---- Context: powered n=600 headline numbers from the full run ----
print("Full-run (n=600) headline numbers for context:")
for k in ["verdict", "H1_modeA_vs_PoT_gap", "H1_modeA_vs_SC_gap",
          "H2_confident_wrong_modeA", "H2_confident_wrong_raw", "H2_confident_wrong_reduction",
          "modeA_resolution_coverage", "modeA_singleton_to_correct_rate", "applicability_verdict"]:
    print(f"  {k:32s} : {headline[k]}")